In [ ]:
import json, requests, time
from collections import Counter
from google.colab import drive, userdata
from datasets import load_dataset

drive.mount('/content/drive')

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
print(f"OpenAI key loaded: {OPENAI_API_KEY[:10]}...{OPENAI_API_KEY[-4:]}")

r = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers={"Authorization": f"Bearer {OPENAI_API_KEY}", "Content-Type": "application/json"},
    json={
        "model": "gpt-4o-mini",
        "messages": [{"role": "user", "content": "Reply with only the word PING"}],
        "max_tokens": 5,
        "temperature": 0.0
    },
    timeout=30
)
print(f"Test status: {r.status_code}")
print(f"Test response: {r.json()['choices'][0]['message']['content'] if r.status_code == 200 else r.text[:300]}")

dataset = load_dataset("GBaker/MedQA-USMLE-4-options")
with open('/content/drive/MyDrive/llama_results.json') as f:
    llama = {r['idx']: r for r in json.load(f)}
with open('/content/drive/MyDrive/bias_labels.json') as f:
    trap_labels_llama = json.load(f)
with open('/content/drive/MyDrive/non_trap_bias_labels.json') as f:
    non_trap_labels_llama = json.load(f)

all_labels_llama = trap_labels_llama + non_trap_labels_llama

all_labels_llama = [b for b in all_labels_llama if b['bias_type'] != 'ERROR']
print(f"\nTotal questions to re-classify with GPT-4o-mini: {len(all_labels_llama)}")
print(f"  Traps: {len(trap_labels_llama)}")
print(f"  Non-traps (excluding errors): {len([b for b in non_trap_labels_llama if b['bias_type'] != 'ERROR'])}")

In [ ]:
def classify_bias_gpt(question, options, gold, wrong_answer, api_key):
    gold_text = options.get(gold, '')
    wrong_text = options.get(wrong_answer, '')
    opts_text = "\n".join([f"{k}: {v}" for k, v in options.items()])

    prompt = f"""You are an expert medical educator. Classify the cognitive bias that caused this AI model error.

Question: {question}

Options:
{opts_text}

Correct answer: {gold} - {gold_text}
Wrong answer chosen: {wrong_answer} - {wrong_text}

Reply with ONLY one of these labels:
- AVAILABILITY_BIAS (over-weighting salient/memorable symptoms)
- ANCHORING_BIAS (over-relying on first piece of information)
- FRAMING_EFFECT (different conclusion from same info presented differently)
- PREMATURE_CLOSURE (stopping reasoning too early)
- OTHER

Single label only, no explanation:"""

    r = requests.post(
        "https://api.openai.com/v1/chat/completions",
        headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
        json={
            "model": "gpt-4o-mini",
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 10,
            "temperature": 0.0
        },
        timeout=30
    )
    if r.status_code != 200:
        return 'ERROR', f"HTTP {r.status_code}: {r.text[:200]}"
    try:
        text = r.json()['choices'][0]['message']['content'].strip().upper()
    except (KeyError, IndexError):
        return 'ERROR', f"Bad shape: {r.text[:200]}"
    for label in ['AVAILABILITY_BIAS', 'ANCHORING_BIAS',
                  'FRAMING_EFFECT', 'PREMATURE_CLOSURE', 'OTHER']:
        if label in text:
            return label, None
    return 'OTHER', None

print(f"Classifying {len(all_labels_llama)} questions with GPT-4o-mini...")
print(f"Model: gpt-4o-mini (DIFFERENT family from Llama-3.3-70B — this is the point)\n")

gpt_labels = []
consecutive_errors = 0

for i, b in enumerate(all_labels_llama):
    idx = b['idx']
    q = dataset['test'][idx]
    llama_pred = llama[idx]['pred']

    label, err = classify_bias_gpt(
        question=q['question'],
        options=q['options'],
        gold=q['answer_idx'],
        wrong_answer=llama_pred,
        api_key=OPENAI_API_KEY
    )

    if label == 'ERROR':
        consecutive_errors += 1
        print(f"  [{i+1}] idx={idx} ERROR: {err}")
        if consecutive_errors >= 5:
            print(f"\n  >>> Aborting: 5 consecutive errors <<<")
            break
    else:
        consecutive_errors = 0

    gpt_labels.append({
        'idx': idx,
        'bias_type': label,
        'llama_label': b['bias_type'],
        'is_trap': b in trap_labels_llama
    })

    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(all_labels_llama)} done...")

    time.sleep(0.2)

with open('/content/drive/MyDrive/bias_labels_gpt4omini.json', 'w') as f:
    json.dump(gpt_labels, f)

valid = [g for g in gpt_labels if g['bias_type'] != 'ERROR']
counts = Counter(g['bias_type'] for g in valid)
total = len(valid)
print(f"\n=== GPT-4o-mini bias distribution (n={total}) ===")
for bias, count in sorted(counts.items(), key=lambda x: -x[1]):
    print(f"  {bias}: {count} ({count/total*100:.1f}%)")

print(f"\nSaved to /content/drive/MyDrive/bias_labels_gpt4omini.json")

In [ ]:
import numpy as np
from collections import Counter
import json

with open('/content/drive/MyDrive/bias_labels_gpt4omini.json') as f:
    gpt_labels = json.load(f)

pairs = [(g['llama_label'], g['bias_type']) for g in gpt_labels
         if g['llama_label'] not in ('ERROR',) and g['bias_type'] not in ('ERROR',)]
print(f"Paired classifications: {len(pairs)}\n")

categories = ['PREMATURE_CLOSURE', 'ANCHORING_BIAS', 'AVAILABILITY_BIAS', 'FRAMING_EFFECT', 'OTHER']

cm = np.zeros((len(categories), len(categories)), dtype=int)
cat2i = {c: i for i, c in enumerate(categories)}
for llama_lbl, gpt_lbl in pairs:
    cm[cat2i[llama_lbl], cat2i[gpt_lbl]] += 1

print("Confusion matrix (rows = Llama-3.3-70B label, cols = GPT-4o-mini label):")
print(f"{'':<22} " + " ".join([f"{c[:6]:>7}" for c in categories]))
for i, c in enumerate(categories):
    print(f"{c:<22} " + " ".join([f"{cm[i,j]:>7}" for j in range(len(categories))]))

n = len(pairs)
agree = sum(1 for l, g in pairs if l == g)
po = agree / n
print(f"\nRaw agreement: {agree}/{n} = {po*100:.1f}%")

llama_marginal = Counter(l for l, g in pairs)
gpt_marginal = Counter(g for l, g in pairs)
print(f"\nMarginal distributions:")
for c in categories:
    lm = llama_marginal.get(c, 0)
    gm = gpt_marginal.get(c, 0)
    print(f"  {c:<22} Llama={lm:>3} ({lm/n*100:>5.1f}%)   GPT={gm:>3} ({gm/n*100:>5.1f}%)")

pe = sum((llama_marginal.get(c, 0)/n) * (gpt_marginal.get(c, 0)/n) for c in categories)
kappa = (po - pe) / (1 - pe) if pe < 1 else 0.0
print(f"\nCohen's kappa:")
print(f"  p_o (observed agreement)  = {po:.4f}")
print(f"  p_e (chance agreement)    = {pe:.4f}")
print(f"  kappa                     = {kappa:.4f}")

k = len([c for c in categories if llama_marginal.get(c,0) > 0 or gpt_marginal.get(c,0) > 0])
pe_gwet = 0.0
for c in categories:
    pi_c = (llama_marginal.get(c, 0) + gpt_marginal.get(c, 0)) / (2*n)
    pe_gwet += pi_c * (1 - pi_c)
pe_gwet = pe_gwet / (k - 1) if k > 1 else 0
ac1 = (po - pe_gwet) / (1 - pe_gwet) if pe_gwet < 1 else 0.0
print(f"\nGwet's AC1 (robust to imbalanced marginals):")
print(f"  AC1 = {ac1:.4f}")

print(f"\nPer-category agreement:")
print(f"{'Category':<22} {'Both yes':<10} {'Both no':<10} {'Llama only':<12} {'GPT only':<10} {'Sensitivity':<12}")
for c in categories:
    both_yes = sum(1 for l, g in pairs if l == c and g == c)
    both_no = sum(1 for l, g in pairs if l != c and g != c)
    llama_only = sum(1 for l, g in pairs if l == c and g != c)
    gpt_only = sum(1 for l, g in pairs if l != c and g == c)

    sens = both_yes / (both_yes + llama_only) if (both_yes + llama_only) > 0 else float('nan')
    print(f"  {c:<22} {both_yes:<10} {both_no:<10} {llama_only:<12} {gpt_only:<10} {sens:<12.3f}")

print(f"\n=== INTERPRETATION ===")
if kappa >= 0.8:
    grade = "ALMOST PERFECT — methodology has very strong inter-rater reliability"
elif kappa >= 0.6:
    grade = "SUBSTANTIAL — methodology has acceptable reliability for publication"
elif kappa >= 0.4:
    grade = "MODERATE — borderline; need to discuss in paper and possibly add human validation"
elif kappa >= 0.2:
    grade = "FAIR — methodology reliability is weak; major paper concern"
else:
    grade = "POOR — methodology is not reliable enough to anchor a paper"

print(f"Cohen's kappa = {kappa:.3f}: {grade}")
print(f"Gwet's AC1   = {ac1:.3f}: more reliable on imbalanced data — this is what reviewers want to see")

In [ ]:
from sklearn.metrics import cohen_kappa_score
llama_binary = [1 if p[0] == 'PREMATURE_CLOSURE' else 0 for p in pairs]
gpt_binary = [1 if p[1] == 'PREMATURE_CLOSURE' else 0 for p in pairs]
kappa_bin = cohen_kappa_score(llama_binary, gpt_binary)
print(f"Binary PC-vs-not Cohen's kappa: {kappa_bin:.3f}")
both_pc = sum(1 for l, g in zip(llama_binary, gpt_binary) if l==1 and g==1)
both_not = sum(1 for l, g in zip(llama_binary, gpt_binary) if l==0 and g==0)
print(f"Raw binary agreement: {(both_pc + both_not)/len(pairs)*100:.1f}%")